Scenario
You are a junior data analyst working for a regional healthcare network that manages multiple clinics. Your team has noticed an increase in missed appointments and inconsistent follow-up care, which is affecting patient outcomes and clinic efficiency. Your manager asks you to [investigate patterns in patient appointment behavior to help identify potential risks and improve scheduling decisions.]

Using the available patient, appointment, provider, and clinic data, your task is to design prompts to guide an AI assistant in generating SQL queries that uncover meaningful patterns. For example, you may need to define what constitutes a "high-risk patient," identify repeat no-show appointments, or analyze appointment trends across clinics or providers. Because the problem is not fully defined, you will need to clarify the business question, decide what data is relevant, and structure your prompts carefully. 

Your goal is to produce SQL that helps the team better understand the problem and supports more informed decisions.

## Initial prompt: 

Task: 

I work for a faux healthcare network. I have noticed an increase in missed appointments and inconsistent follow-up care, which is affecting patient outcomes and clinic efficiency. Investigate patterns in patient appointment behavior to help identify potential risks and improve scheduling decisions.


The schema is as follows:

**Table: patients**


- patient\_id — patient identifier
- patient\_name — patient full name
- date\_of\_birth — patient date of birth
- city — patient city
- insurance\_type — patient insurance category

**Appointments table**

Contains appointment activity and status information.

**Table: appointments**

- appointment\_id — appointment identifier
- patient\_id — links to patients.patient\_id
- provider\_id — links to providers.provider\_id
- clinic\_id — links to clinics.clinic\_id
- appointment\_date — date of appointment
- appointment\_status — appointment outcome, such as completed, canceled, or no\_show
- appointment\_type — type of appointment, such as checkup, consult, screening, or follow\_up

**Reporting Context table**

Contains the relevant dates

**Table: reporting\_context**

- context\_id
- reporting\_date
- data\_start\_date
- data\_end\_date
- recent\_30\_day\_start
- recent\_60\_day\_start
- recent\_3\_month\_start
- recent\_6\_month\_start
- primary\_date\_column
- note

The reporting\_context table does not join to the appointment tables. It provides fixed date boundaries for this fictional dataset. Use reporting\_context instead of date('now') when defining recent appointment windows.

**Providers table**

Contains provider and specialty information.

Table: providers

- provider\_id — provider identifier
- provider\_name — provider name
- specialty — provider specialty

**Clinics table**

Contains clinic location information.

Table: clinics

- clinic\_id — clinic identifier
- clinic\_name — clinic name
- city — clinic city
- region — clinic region

**Relationship Reference**

patients.patient\_id = appointments.patient\_id

providers.provider\_id = appointments.provider\_id

clinics.clinic\_id = appointments.clinic\_id



Constraints:

Strictly use the above schema when creating prompt responses. Produce a usable SQL prompt for each of the following questions I will ask.



Output: Produce a response that I can past within my SQL terminal

## Response

Got it. I’ll follow the provided schema exactly and return only runnable SQL that you can paste into your SQL terminal.

Send me the first question.

In [52]:
import sqlite3
import pandas as pd

DB_FILE = "healthcare_operations.db"
conn = sqlite3.connect(DB_FILE)
print(f"Connected to {DB_FILE}")

Connected to healthcare_operations.db


Audit the categorical/text fields so I can spot any incosistencies in field nams, values etc

In [53]:
unique_appointment_values = pd.read_sql("""
SELECT 'appointment_status' AS column_name,
       appointment_status AS value,
       COUNT(*) AS count
FROM appointments
GROUP BY appointment_status

UNION ALL

SELECT 'appointment_type',
       appointment_type,
       COUNT(*)
FROM appointments
GROUP BY appointment_type

ORDER BY column_name, value;
""", conn)

display(unique_appointment_values)

,column_name,value,count
0,appointment_status,canceled,10
1,appointment_status,completed,87
2,appointment_status,no_show,23
3,appointment_type,checkup,29
4,appointment_type,consult,32
5,appointment_type,follow_up,36
6,appointment_type,screening,23


In [54]:
unique_patient_values = pd.read_sql("""
SELECT 'patient_name' AS column_name,
       patient_name AS value,
       COUNT(*) AS count
FROM patients
GROUP BY patient_name

UNION ALL

SELECT 'city',
       city,
       COUNT(*)
FROM patients
GROUP BY city

UNION ALL

SELECT 'insurance_type',
       insurance_type,
       COUNT(*)
FROM patients
GROUP BY insurance_type

ORDER BY column_name, value;
""", conn)

# display(unique_patient_values)

Using the provided SQLite database schema, write a `pd.read_sql()` query that shows the relationship between each patient and their no-show appointment behavior.

For each patient, return:

* Patient ID
* Patient name
* Total number of appointments
* Total number of no-show appointments
* No-show rate as a percentage
* Reporting date
* Reporting period start date
* Reporting period end date

Use the `patients`, `appointments`, and `reporting_context` tables and their defined relationships. Use the reporting boundaries provided by `reporting_context` rather than `date('now')`.

For `appointment_status`, treat the standardized value `no_show` as a no-show. Calculate the no-show rate as:

no-show appointments / total appointments × 100

Handle patients with zero appointments safely and order the results by highest no-show count and then highest no-show rate.

The SQL must be SQLite-compatible and formatted for use with `pd.read_sql(..., conn)`. Do not invent tables or columns outside the provided schema.


In [55]:
patient_no_shows = pd.read_sql("""
SELECT
    p.patient_id,
    p.patient_name,
    COUNT(a.appointment_id) AS total_appointments,
    SUM(
        CASE
            WHEN LOWER(TRIM(a.appointment_status)) = 'no_show'
            THEN 1
            ELSE 0
        END
    ) AS no_show_count,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN LOWER(TRIM(a.appointment_status)) = 'no_show'
                THEN 1
                ELSE 0
            END
        ) / NULLIF(COUNT(a.appointment_id), 0),
        2
    ) AS no_show_rate,
    rc.reporting_date,
    rc.data_start_date,
    rc.data_end_date
FROM patients p
LEFT JOIN appointments a
    ON p.patient_id = a.patient_id
CROSS JOIN reporting_context rc
WHERE a.appointment_date BETWEEN rc.data_start_date AND rc.data_end_date
GROUP BY
    p.patient_id,
    p.patient_name,
    rc.reporting_date,
    rc.data_start_date,
    rc.data_end_date
ORDER BY no_show_count DESC, no_show_rate DESC;
""", conn)

display(patient_no_shows)

,patient_id,patient_name,total_appointments,no_show_count,no_show_rate,reporting_date,data_start_date,data_end_date
0,8,Hannah Davis,3,2,66.67,2025-06-29,2025-01-06,2025-06-29
1,17,Quinn Lewis,3,2,66.67,2025-06-29,2025-01-06,2025-06-29
2,39,Maya Stewart,3,2,66.67,2025-06-29,2025-01-06,2025-06-29
3,46,Thomas Bell,3,2,66.67,2025-06-29,2025-01-06,2025-06-29
4,3,Carlos Rivera,4,2,50.00,2025-06-29,2025-01-06,2025-06-29
5,28,Brandon Carter,4,2,50.00,2025-06-29,2025-01-06,2025-06-29
6,12,Lily Thomas,2,1,50.00,2025-06-29,2025-01-06,2025-06-29
7,20,Tyler Allen,2,1,50.00,2025-06-29,2025-01-06,2025-06-29
8,23,Wendy King,2,1,50.00,2025-06-29,2025-01-06,2025-06-29
9,32,Felix Turner,2,1,50.00,2025-06-29,2025-01-06,2025-06-29


For the above response I had the fix now_show as it had an incorrect value.